In [45]:
from google.colab import userdata
import os

openrouter_key = userdata.get("OPENROUTER_API_KEY")
tavily_key = userdata.get("TAVILY_API_KEY")

assert openrouter_key, "OPENROUTER_API_KEY not found in Colab Secrets"
assert tavily_key, "TAVILY_API_KEY not found in Colab Secrets"

os.environ["OPENROUTER_API_KEY"] = openrouter_key.strip()
os.environ["TAVILY_API_KEY"] = tavily_key.strip()

print("OpenRouter loaded:", os.environ["OPENROUTER_API_KEY"][:12])
print("Tavily loaded:", os.environ["TAVILY_API_KEY"][:12])

OpenRouter loaded: sk-or-v1-961
Tavily loaded: tvly-dev-3rU


In [31]:
!pip install -U langchain langchain_openai langchain_community requests python-dotenv beautifulsoup4

In [32]:
from google.colab import userdata
import os

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [33]:
import os
import requests
from bs4 import BeautifulSoup

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from langchain_community.tools.tavily_search import TavilySearchResults

In [34]:
internet_search = TavilySearchResults()

In [35]:
!pip install -U langchain-tavily

In [36]:
from langchain_tavily import TavilySearch

internet_search = TavilySearch(max_results=3)

In [37]:
internet_search.invoke({"query": "AI in education"})

{'query': 'AI in education',
 'response_time': 0.7,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.educationnext.org/a-i-in-education-leap-into-new-era-machine-intelligence-carries-risks-challenges-promises/',
   'title': 'AI in Education - Education Next',
   'content': 'Microsoft researchers suggest that newer systems “exhibit more general intelligence than previous AI models” and are coming “strikingly close to human-level performance.” While some observers question those conclusions, the AI systems display an increasing ability to generate coherent and contextually appropriate responses, make connections between different pieces of information, and engage in reasoning processes such as inference, deduction, and analogy. AI can also provide educators with recommendations to meet student needs and help teachers reflect, plan, and improve their practice. ***Privacy concerns.*** When students or educators interact with generative-AI tool

In [38]:
@tool
def fetch_url(url: str) -> str:
    """Fetch readable text content from a URL."""

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)

        if response.status_code != 200:
            return f"Failed to fetch page: {url}"

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)

        return text[:12000]

    except Exception as e:
        return f"Error fetching {url}: {str(e)}"

In [39]:
test_url = "https://en.wikipedia.org/wiki/Artificial_intelligence_in_education"

print(fetch_url.invoke({"url": test_url})[:1000])

Artificial intelligence in education - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main page Contents Current events Random article About Wikipedia Contact us Contribute Help Learn to edit Community portal Recent changes Upload file Special pages Search Search Appearance Donate Create account Log in Personal tools Donate Create account Log in Contents move to sidebar hide (Top) 1 History 2 Theory Toggle Theory subsection 2.1 Three paradigms of AIEd 2.2 Socio-technical imaginaries 2.3 Post-humanist and new materialist perspectives 3 Applications Toggle Applications subsection 3.1 AI-based tutoring systems 3.2 Generative AI 3.3 Emotional AI 4 Perspectives Toggle Perspectives subsection 4.1 Commercial perspectives 4.2 Institutional perspectives 4.3 Student perspectives 5 Problems Toggle Problems subsection 5.1 Over-reliance, inaccuracy, and academic integrity 5.2 Accessibility 5.3 Bias 5.4 Data privacy and intellectual property 5.5 Invisible labour and en

In [40]:
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

In [41]:
response = model.invoke("Say hello in one short sentence.")
print(response.content)

Hello!


In [42]:
system_prompt = """
You are an education research agent.

You must follow this exact process:
1. Read the user's question carefully.
2. Use internet_search to find relevant information about AI in education.
3. Select exactly the top 3 URLs from the search results.
4. Call fetch_url for each of those 3 URLs at most once.
5. Use only successfully fetched page contents in the final answer.
6. If one URL fails, skip it and continue with the remaining successful sources.
7. Do not retry the same failed URL repeatedly.
8. Do not cite any URL that failed to fetch successfully.
9. Write a clear answer about the benefits, risks, and practical impact of AI in education.
10. Include only the URLs that were fetched successfully in the final answer.
"""

In [43]:
agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=system_prompt,
)

In [46]:
question = "What are the benefits and risks of artificial intelligence in education?"

response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

print(response["messages"][-1].content)

Artificial intelligence is transforming education in ways that can enhance learning while also introducing new challenges.  

**Key Benefits**  
- **Personalized learning** – AI tailors content and pacing to each student’s needs, helping diverse learners stay engaged and progress at their own speed.  
- **Automation of routine tasks** – Activities such as grading, scheduling, and resource management are streamlined, freeing teachers to spend more time on instruction and student interaction.  
- **Data‑driven insights** – Advanced analytics reveal patterns in student performance, enabling early intervention and more informed instructional decisions.  
- **Innovative instructional tools** – Adaptive platforms, simulations, and AI‑powered writing assistants create immersive, interactive experiences and support multilingual or special‑needs students.  
- **Global accessibility** – Remote‑learning capabilities break down geographic barriers, extending quality education to underserved region